<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Solution: Regression intuition

**[C05-M02-L01 · Regression intuition](https://learning.nextia-ai.com/courses/ml/m02/regression-intuition/)** · C05, Machine Learning: From Problem to Reliable Model · Module 2, lesson 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** fit a straight-line model that predicts a number, read its residuals, and check whether it works on days it has not seen.

| | |
|---|---|
| **Time** | About 55 minutes |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6 and scikit-learn 1.9.1. |
| **You should know** | The loss as a mean squared error, from [Loss and gradients](https://learning.nextia-ai.com/courses/math/m04/loss-and-gradients/). Regression as a learning task, from [Choose the learning task](https://learning.nextia-ai.com/courses/ml/m01/choose-the-learning-task/). |
| **You do not need** | The local project. The setup below downloads the data here. |
| **Tested** | 2026-10-07, Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m02/regression-intuition/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M02-L01-regression-intuition/regression-intuition-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the C05 practice data (about 2.5 MB) into a folder `ticket-model/data`, and checks each file's checksum. It is the same data that `get_data.py` downloads in [Define a baseline](https://learning.nextia-ai.com/courses/ml/m01/define-a-baseline/). It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 16939 rows (1983 escalated), valid 1179 (183), test 1002 (157).` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-model` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the C05 data into ticket-model/data/ and check it.
# Standard library only: the same steps as get_data.py.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/"
FILES = {
    "train.csv": "63b8732a28e891189cc28a9bcd505a093a9c4c51c01f3d0e1fba6d4959242d53",
    "valid.csv": "3604df83ea2ec3cec70c2eedfc7e95cc6d5cb82ccfc3b5cf565e614783e2bde2",
    "test.csv": "60e89f75e4d7d59ccfbda49ca719ce7f8e8d97c0380256d633ab240d546771ec",
    "extra_features.csv": "03c6da28bcb347982aa70a7c72b869771a6cb055654d629da924512c6fd56563",
    "daily_tickets.csv": "7deba76b112b4133a5d16b74be7e2b331ce28c7064e7fd8dd471768e5f1ba360",
    "july_tickets.csv": "10dffc093993341a361172d0aa6ea0e7c06521f140cc2a80ee90c713361f7c99",
    "july_labels.csv": "3163c57e398db6fd1c19e505548d0fa1464821ef96c77901a759abbd3a8605fa",
}
PROJECT_FILES = []
HERE = Path.cwd()
# The notebook works in its own folder ticket-model/, marked with a file, so
# it never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("ticket-model").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-model and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)
for name in PROJECT_FILES:
    fetch(name, LABS + "project/" + name, PROJECT / name)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
counts = {}
for part in ("train", "valid", "test"):
    rows = Path(f"data/{part}.csv").read_text(encoding="utf-8").splitlines()[1:]
    counts[part] = (len(rows), sum(row.endswith(",1") for row in rows))
print("Ready: train {} rows ({} escalated), valid {} ({}), test {} ({}).".format(
    *counts["train"], *counts["valid"], *counts["test"]))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd` and numpy as `np`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A line through seven days

Grace, the support lead, plans how many agents work each day. She asks Priya, the data scientist: how many tickets will arrive tomorrow? The answer is a number, so this is a **regression** task.

Start with one week. Run the next cell. It reads the first seven days of `data/daily_tickets.csv`.

In [ ]:
import pandas as pd

daily = pd.read_csv("data/daily_tickets.csv", parse_dates=["date"])
week = daily.head(7)[["date", "weekday", "is_weekend", "tickets"]]
print(week)

> **Check.** You should see 1 to 7 July 2024: 33, 33, 29, 36 and 29 tickets on the weekdays, then 13 and 9 at the weekend.

Two models for this week:

- **Constant:** predict the mean of the week every day: 182 ÷ 7 = 26 tickets.
- **Line:** predict `tickets = a + b × is_weekend`, with `a = 32` and `b = −21`. A weekday gets 32; a weekend day gets 32 − 21 = 11.

A **residual** is the actual value minus the predicted value. A positive residual means the model predicted too few. The **loss** is the mean squared error (MSE): the mean of the squared residuals, as in C03.

> **Predict.** Which model has the lower MSE? By about how much? Then run the cell.

In [ ]:
week = week.assign(
    constant=26,
    line=32 - 21 * week["is_weekend"],
)
week = week.assign(
    residual_constant=week["tickets"] - week["constant"],
    residual_line=week["tickets"] - week["line"],
)
print(week[["weekday", "tickets", "constant", "residual_constant", "line", "residual_line"]])
print("MSE constant:", round((week["residual_constant"] ** 2).mean(), 2))
print("MSE line:", round((week["residual_line"] ** 2).mean(), 2))

> **Check.** You should see `MSE constant: 96.29` and `MSE line: 6.29`.

Check one row by hand. Saturday has 13 tickets. The line predicts 11, so the residual is 13 − 11 = 2, and its square is 4. The constant predicts 26, so its residual is −13, and its square is 169.

**Least squares** means: choose `a` and `b` so that the MSE is as small as possible. For a feature that is only 0 or 1, the best `a` is the mean of the weekdays (160 ÷ 5 = 32). The best `a + b` is the mean of the weekend days (22 ÷ 2 = 11).

> **Your turn.** Is 32 − 21 really the best line? Compute the MSE of another line, `tickets = 33 − 22 × is_weekend`, on the same week. Put it in `mse_other`, rounded to 2 decimals. Then run the check cell.

In [ ]:
predicted = 33 - 22 * week["is_weekend"]
mse_other = round(((week["tickets"] - predicted) ** 2).mean(), 2)
print(mse_other)

In [ ]:
expect_hash('mse_other', mse_other, '0de7cf39e1e63661')

The other line has a higher MSE (7.0 against 6.29). Any line other than 32 − 21 has a higher MSE on this week.

## 2. scikit-learn finds the same line

`LinearRegression` from scikit-learn does least squares for you. `fit` finds `a` (`intercept_`) and `b` (`coef_`). Run the cell.

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(week[["is_weekend"]], week["tickets"])
print("a (intercept_):", round(model.intercept_, 2))
print("b (coef_):", model.coef_.round(2))
print("predictions:", model.predict(week[["is_weekend"]]).round(2))

> **Check.** You should see `a (intercept_): 32.0`, `b (coef_): [-21.]` and the predictions 32 for the weekdays and 11 for the weekend.

The features go in as a table (`week[["is_weekend"]]`, two pairs of brackets), even when there is only one. The target is one column (`week["tickets"]`).

## 3. The real data: train before May, test on May and June

Priya uses all 730 days. She trains on the days before 1 May 2026 and tests on May and June 2026, the busy garden season. A split by time copies real use: the model predicts days that come after the days it learned from.

The month is a number from 1 to 12, but month 12 is not "twelve times" month 1. So the cell turns it into twelve columns of 0 and 1, one for each month (`month_1` … `month_12`). This is safe here, because the twelve months are fixed and known in advance. Run the cell.

In [ ]:
daily = daily.join(pd.get_dummies(daily["month"], prefix="month", dtype=int))
MONTHS = [f"month_{m}" for m in range(1, 13)]

train = daily[daily["date"] < "2026-05-01"].copy()
test = daily[daily["date"] >= "2026-05-01"].copy()
print(len(train), "train days,", len(test), "test days")
print("mean tickets per day in train:", round(train["tickets"].mean(), 2))

> **Check.** You should see `669 train days, 61 test days` and `mean tickets per day in train: 25.32`.

The next cell defines `report`. It prints two error measures on train and on test:

- **MAE** (mean absolute error): the mean size of the residuals, in tickets.
- **RMSE** (root mean squared error): the square root of the MSE, also in tickets. It grows more when some errors are large.

It also prints the **mean test residual**: positive means the model predicts too few on average. Module 4 explains the metrics fully. Run the cell. It prints nothing.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error


def report(name, train_predicted, test_predicted):
    """Print MAE and RMSE on train and test, and the mean test residual."""
    parts = []
    for part, predicted in [(train, train_predicted), (test, test_predicted)]:
        mae = mean_absolute_error(part["tickets"], predicted)
        rmse = mean_squared_error(part["tickets"], predicted) ** 0.5
        parts.append(f"MAE {mae:.2f}, RMSE {rmse:.2f}")
    residual = (test["tickets"] - test_predicted).mean()
    print(f"{name}: train {parts[0]} | test {parts[1]} | mean test residual {residual:+.2f}")

First the simplest model: predict the train mean, 25.32, every day. It is the baseline that every other model must beat.

> **Predict.** Will the mean test residual be positive or negative? Then run the cell.

In [ ]:
train_mean = train["tickets"].mean()
report("constant", [train_mean] * len(train), [train_mean] * len(test))

> **Check.** You should see `constant: train MAE 8.52, RMSE 10.43 | test MAE 14.95, RMSE 17.27 | mean test residual +12.27`.

The test error is much larger than the train error. The residual is positive: May and June are busy, and the constant predicts too few.

Now the weekend line on the real data. Run the cell.

In [ ]:
weekend = LinearRegression().fit(train[["is_weekend"]], train["tickets"])
print(f"tickets = {weekend.intercept_:.2f} + ({weekend.coef_[0]:.2f}) x is_weekend")
report("is_weekend", weekend.predict(train[["is_weekend"]]), weekend.predict(test[["is_weekend"]]))

> **Check.** You should see `tickets = 29.99 + (-16.45) x is_weekend` and `is_weekend: train MAE 5.86, RMSE 7.33 | test MAE 12.54, RMSE 14.08 | mean test residual +12.45`.

The weekend helps on train, but the test error stays large. The model knows nothing about the season. Add the twelve month columns. Run the cell.

In [ ]:
COLUMNS = ["is_weekend"] + MONTHS
season = LinearRegression().fit(train[COLUMNS], train["tickets"])
train["predicted"] = season.predict(train[COLUMNS])
test["predicted"] = season.predict(test[COLUMNS])
report("is_weekend + month", train["predicted"], test["predicted"])

> **Check.** You should see `is_weekend + month: train MAE 4.41, RMSE 5.64 | test MAE 6.45, RMSE 8.19 | mean test residual +4.98`.

Much better: the test MAE falls from 14.95 to 6.45. But the residual is still about +5 tickets a day. The model predicts too few, every day, on average.

## 4. Read the residuals

A model **fits** the train days when its train error is small. It **generalizes** when its error stays small on days it has not seen. Here the test MAE (6.45) is larger than the train MAE (4.41). The residuals show why.

Run the next cell. It draws the last four months: the actual tickets and the model.

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt

recent = pd.concat([train, test]).query("date >= '2026-03-01'")
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(recent["date"], recent["tickets"], label="actual tickets")
ax.plot(recent["date"], recent["predicted"], label="model (is_weekend + month)", linestyle="--")
ax.axvline(pd.Timestamp("2026-05-01"), color="grey", linewidth=1)
ax.set_xlabel("Day (test starts on 1 May 2026, grey line)")
ax.set_ylabel("Tickets per day")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax.legend()
plt.show()

> **Check.** You should see the weekly pattern: five high days and two low days. After 1 May, the actual line is above the model on most days.

> **Your turn.** Compute the mean residual of the test days for each month (5 = May, 6 = June). Put the result in `by_month`, as a pandas Series rounded to 2 decimals. Use `test["tickets"] - test["predicted"]`, then `groupby(test["month"])`. Run the cell, then the check cell.

In [ ]:
test["residual"] = test["tickets"] - test["predicted"]
by_month = test["residual"].groupby(test["month"]).mean().round(2)
print(by_month)

In [ ]:
expect_hash('by_month', by_month, '5ef10e9611ac2fa4')

Both test months are under-predicted by about 5 tickets a day. Now look at the train residuals by month. Run the cell.

In [ ]:
train["residual"] = train["tickets"] - train["predicted"]
print(train["residual"].groupby(train["date"].dt.to_period("M")).mean().round(2).to_string())

> **Check.** You should see negative means from July 2024 to April 2025, for example −3.18 in July 2024. May and June 2025 show 0.00. From July 2025 to April 2026 the means are positive, for example +3.08 in April 2026.

The pattern is clear: the second year is busier than the first. Larkfield has more customers each month. The model gives every July the same value, so it predicts too many in the first year and too few in the second. May and June 2025 are exact because train has only one May and one June: the model copies them. Then it predicts May 2026 from May 2025, and the shop has grown since. No column says "time", so the model cannot see the growth.

## 5. Change one thing: add a trend

Add one column, `days`: the number of days since 1 July 2024. With it, the model can learn a steady growth.

> **Predict.** Will the mean test residual go up, go down, or stay near +5?

> **Your turn.** Add `"days"` to the list `trend_columns`, then run the cell and the check cell.

In [ ]:
daily["days"] = (daily["date"] - pd.Timestamp("2024-07-01")).dt.days
train["days"] = daily.loc[train.index, "days"]
test["days"] = daily.loc[test.index, "days"]

trend_columns = ["is_weekend"] + MONTHS + ["days"]  # add "days" to this list
trend = LinearRegression().fit(train[trend_columns], train["tickets"])
trend_test = trend.predict(test[trend_columns])
report("with trend", trend.predict(train[trend_columns]), trend_test)
trend_result = [round(mean_absolute_error(test["tickets"], trend_test), 2),
                round((test["tickets"] - trend_test).mean(), 2)]

In [ ]:
expect_hash('trend_result', trend_result, '0d101acb8e490007')

With the trend, you should see `with trend: train MAE 4.09, RMSE 5.22 | test MAE 5.01, RMSE 6.52 | mean test residual +0.54`. Run the next cell to see the learned growth.

In [ ]:
if "days" in trend_columns:
    print("tickets per day added each day:", round(trend.coef_[trend_columns.index("days")], 4))
else:
    print('Add "days" to trend_columns first, then run the two cells above again.')

> **Check.** You should see `tickets per day added each day: 0.0122`: about 4.5 more tickets per day each year (0.0122 × 365).

The mean test residual falls from +4.98 to +0.54. A trend is also a risk: the model assumes that the growth continues at the same rate, forever. Check it again each month.

## 6. Failure case: a random test set

Tomás, the AI engineer, takes 61 days at random as the test set, instead of May and June.

> **Predict.** Will his test MAE be higher or lower than 6.45? Then run the cell.

In [ ]:
random_test = daily.sample(n=61, random_state=0)
random_train = daily.drop(random_test.index)
model = LinearRegression().fit(random_train[COLUMNS], random_train["tickets"])
predicted = model.predict(random_test[COLUMNS])
print("random test MAE:", round(mean_absolute_error(random_test["tickets"], predicted), 2))
print("random test mean residual:", round((random_test["tickets"] - predicted).mean(), 2))

> **Check.** You should see `random test MAE: 4.38` and `random test mean residual: -1.1`.

The random test looks better (4.38 instead of 6.45), and it shows no bias. It is wrong. The random test days sit between train days of the same months, so the model has already seen their neighbours. The growth stays hidden. Grace would staff May and June for about 5 tickets a day too few. **Split by time when the model predicts the future**, as in C04's [Choose a split](https://learning.nextia-ai.com/courses/sql/m05/choose-a-split/).

## Check your understanding and recap

Answer the **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/ml/m02/regression-intuition/#knowledge-check). They count toward your certificate when you are signed in and enrolled.

In this lesson you fitted lines by least squares, first by hand on one week, then with `LinearRegression`. You read the residuals. The season model fits train well (MAE 4.41). But it under-predicts May and June by about 5 tickets a day (MAE 6.45), because the shop grows. A trend column removes most of that bias (+0.54). A random test set hides the problem.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Classification intuition](https://learning.nextia-ai.com/courses/ml/m02/classification-intuition/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m02/regression-intuition/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.